# Getting Started


This documentation is intended to explain how to compute time conversions and offsets using `timescale`.

## Time

The {py:mod}`time <timescale.time>` module can convert different time formats to the necessary time format of a given program.
It can also parse date strings describing the units and epoch of relative times, or the calendar date of measurement for geotiff formats.
`timescale` keeps updated tables of leap seconds for converting from GPS, LORAN and TAI times.

### Standards

- **TAI time**: International Atomic Time which is computed as the weighted average of several hundred atomic clocks.
- **UTC time**: Coordinated Universal Time which is [periodically adjusted](https://www.nist.gov/pml/time-and-frequency-division/leap-seconds-faqs) to account for the difference between the definition of the second and the rotation of Earth.
- **GPS time**: Atomic timing system for the Global Positioning System constellation of satellites monitored by the United States Naval Observatory (USNO). GPS time and UTC time were equal on January 6, 1980. TAI time is ahead of GPS time by 19 seconds.
- **LORAN time**: Atomic timing system for the Loran-C chain transmitter sites used in terrestrial radionavigation. LORAN time and UTC time were equal on January 1, 1958. TAI time is ahead of LORAN time by 10 seconds.

In [ ]:
import datetime
import timescale
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# adjust style
facecolor = "#fcfcfc"
plt.rcParams["figure.facecolor"] = facecolor
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Lato"]

# build a monthly time axis for plotting
today = datetime.date.today().isoformat()
ts = timescale.from_range("1972-01-01", today, 1, "D")
dt = ts.to_datetime()

# create figure
fig, ax = plt.subplots(figsize=(8, 4.5))
# add differences between timescales
ax.plot(
    dt,
    np.zeros_like(ts.utc),
    color="0.4",
    lw=1.5,
    label="UTC",
    zorder=0,
)
ax.plot(
    dt,
    ts.ut1_utc,
    color="red",
    lw=1.5,
    label="UT1\u2013UTC",
    zorder=2,
)
ax.plot(
    dt,
    ts.gps_utc,
    color="dodgerblue",
    lw=1.5,
    label="GPS\u2013UTC",
    zorder=1,
)
ax.plot(
    dt,
    ts.loran_utc,
    color="mediumseagreen",
    lw=1.5,
    label="LORAN\u2013UTC",
)
ax.plot(
    dt,
    ts.tai_utc,
    color="darkorchid",
    lw=1.5,
    label="TAI\u2013UTC",
)
# show maximum range of UT1 variability
ax.axhline(0.9, color="k", lw=0.7, ls="--")
ax.axhline(-0.9, color="k", lw=0.7, ls="--")
ax.text(
    dt[-100],
    1.0,
    r"$\pm 0.9$s DUT1 limit",
    color="k",
    ha="right",
    va="bottom",
    fontsize=9,
)
# add legend
lgd = ax.legend(frameon=False)
for line in lgd.get_lines():
    line.set_linewidth(6)
# add labels
ax.set_ylabel("Offset relative to UTC [s]")
ax.set_xlabel("Time [yr]")
# set limits and tick locations
ax.set_xlim(dt[0], dt[-1])
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# set layout
fig.tight_layout()
plt.show()

### Dynamic Time

`timescale` also keeps updated tables of delta times for converting between dynamic (TT) and universal (UT1) times.
Delta times (TT - UT1) are the differences between Dynamic Time (TT) and Universal Time (UT1) {cite:p}`Meeus:1991vh`.
Universal Time (UT1) is based on the rotation of the Earth, which varies irregularly, and so UT1 is adjusted periodically.
Dynamic Time (TT) is a uniform, monotonically increasing time standard based on atomic clocks that is used for the accurate calculation of celestial mechanics, orbits and ephemerides.
Delta times can be added to Universal Time (UT1) values to convert to Dynamic Time (TT) values.

```{tip}    
See the [project background on time](https://pytmd.readthedocs.io/en/latest/background/Time.html) for more information on time standards and scales
```

## Functionality

{py:class}`Timescale <timescale.time.Timescale>` objects can be used to convert between date and time formats.
There are a few different ways to create a {py:class}`Timescale <timescale.time.Timescale>` object:

1. Range of dates  
2. Delta times  
3. `datetime` objects  
4. Calendar dates  
5. Julian dates  

In [ ]:
# range of dates
ts = timescale.from_range("2018-01-01", "2018-12-31", 1)

# delta times
delta_time = np.arange(365) * timescale.time._to_sec["day"]
ts = timescale.from_deltatime(delta_time, epoch=(2018, 1, 1, 0, 0, 0))

# datetime objects
date = datetime.datetime(2018, 1, 1, 0, 0, 0)
ts = timescale.from_datetime(date)

# calendar dates
year = 2018
month = 1
day = 1 + np.arange(31)
ts = timescale.from_calendar(year, month, day)

# Julian dates
JD = np.arange(2458119.5, 2458150.5, 1)
ts = timescale.from_julian(JD)

# inspect timescale
ts

{py:class}`Timescale <timescale.time.Timescale>` objects can be used to convert epochs, convert to different time standards, convert to `datetime` arrays, and other time conversions.

```{tip}    
See the {ref}`API Reference <api-time>` for more details on the capabilities
```